In [14]:
import requests
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("OPEN_ROUTER_KEY")

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
} # здесь данные для авторизации запроса

def generate_text(prompt, model):
    # отправляем запрос в openrouter для общения в формате чата 
    url = "https://openrouter.ai/api/v1/chat/completions" 
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}], # роль user - инструкции от пользователя
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()["choices"][0]["message"]["content"] # парсим ответ

prompt = "Анекдот про слона."
print(generate_text(prompt, model="openrouter/owl-alpha"))

Заходит слон в бар. Бармен спрашивает:
— Почему ты такой грустный?
Слон отвечает:
— У меня проблемы с памятью... я всё забываю!
Бармен:
— Ну ты же слон, у тебя должна быть хорошая память!
Слон:
— Вот именно! Я вчера забыл, что у меня сегодня день рождения!


# Задание 1
На локальной машине напишите код запроса, аналогичного запросу выше, с помощью библиотеки OpenAI. Сравните работу DeepSeek с работой другой модели ― например, `qwen/qwen-2.5-72b-instruct:free` (следите за тарификацией!).

Для референса используйте документацию: https://developers.openai.com/api/reference/python/.

In [29]:
import openai
from dotenv import load_dotenv
import os

load_dotenv()

client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY")
)

response = client.chat.completions.create(
    model="google/gemma-4-31b-it:free",
    messages=[
        {"role": "user", "content": "Зачем слонам крылья kfc?"}
    ]
)

print(response.choices[0].message.content)

Эта фраза звучит как абсурдная шутка, мем или загадка с подвохом, потому что в реальности **у слонов нет крыльев**, а **KFC** — это сеть ресторанов быстрого питания, которая продает куриные крылышки.

Скорее всего, здесь смешались три разные вещи:
1. **Слоны** (огромные животные).
2. **Крылья** (которые слонам не нужны).
3. **KFC** (где продают крылышки).

Если это часть какого-то локального мема или шутки, то ответ может быть таким: **«Чтобы долететь до ближайшего KFC за порцией крылышек!»** 🍗🐘

А если серьезно — это просто бессмысленная фраза (сюрреализм).


In [31]:
import torch
import torch.nn as nn
from transformers import GPT2Config, GPT2Model, GPT2Tokenizer
from torch.nn import CrossEntropyLoss

# Загружаем предобученный токенизатор GPT-2 от модели openai-community/gpt2
tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2')

# Создаём конфиг для GPT2Model
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,    # размер словаря (=50257 для 'gpt2')
    n_positions=512,                    # максимальная длина позиции (контекстный window)
    n_ctx=512,                          # то же, что n_positions (макс. размер входа)
    n_embd=128,                         # размер эмбеддингов (по умолчанию у gpt2 — 768)
    n_layer=1,                          # число слоёв декодера (по умолчанию у gpt2 — 12)
    n_head=1,                           # число голов в механизме внимания (по умолчанию — 12)
    activation_function="gelu_new",     # функция активации в feed-forward
    resid_pdrop=0.1,                    # dropout для резидуальных соединений
    embd_pdrop=0.1,                     # dropout после суммирования эмбеддингов
    attn_pdrop=0.1,                     # dropout внутри self-attention
    layer_norm_epsilon=1e-5,            # eps для слоя нормализации
    initializer_range=0.02,             # стандартное отклонение для инициализации весов
    bos_token_id=tokenizer.bos_token_id,# id токена начала последовательности
    eos_token_id=tokenizer.eos_token_id # id токена конца последовательности
)

# Инициализируем модель GPT2Model (без LM-головы)
gpt2 = GPT2Model(config)

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
# Извлекаем веса из Embedding слоя
embedding_weight = gpt2.get_input_embeddings().weight  # shape: (vocab_size, n_embd)

# Добавляем LM голову из размерности скрытого слоя в размер словаря
lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
# из-за особенностей хранения весов nn.Linear в PyTorch матрицу не нужно транспонировать
lm_head.weight = embedding_weight

In [33]:
# Пример данных
texts = [
    "Hello, how are you?",
    "This is a test sentence"
]

# Зададим пэддинг-токен как токен конца
tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token) # выведет <|endoftext|>

<|endoftext|>


In [34]:
# Добавим к каждому предложению токен конца
texts = [text + '<|endoftext|>' for text in texts]

# Токенизируем и приведём к одинаковой длине
encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

input_ids = encodings['input_ids'] # shape: (2, 7)
attention_mask = encodings['attention_mask']

print("input_ids")
print(input_ids)

print("attention_mask")
print(attention_mask)

print("Decoded inputs:")
print(tokenizer.batch_decode(input_ids))


input_ids
tensor([[15496,    11,   703,   389,   345,    30, 50256],
        [ 1212,   318,   257,  1332,  6827, 50256, 50256]])
attention_mask
tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0]])
Decoded inputs:
['Hello, how are you?<|endoftext|>', 'This is a test sentence<|endoftext|><|endoftext|>']


In [35]:
# Forward pass через GPT2Model
# outputs.last_hidden_state: (2, 7, 128)
outputs = gpt2(input_ids=input_ids, attention_mask=attention_mask)
hidden_states = outputs.last_hidden_state

# Линейная проекция для логитов по словарю
# logits: (2, 7, vocab_size)
logits = lm_head(hidden_states)

targets = input_ids[:, 1:].contiguous()
targets_attention_mask = attention_mask[:, 1:]
logits = logits[:, :-1, :].contiguous()

# входные токены
print(tokenizer.decode(input_ids[0, :-1]))

# таргет
print(tokenizer.decode(targets[0]))

Hello, how are you?
, how are you?<|endoftext|>


In [36]:
targets[targets_attention_mask == 0] = -100

logits = logits.view(-1, config.vocab_size) 
targets = targets.view(-1)

# Функция потерь для мультиклассовой классификации
loss_fn = CrossEntropyLoss()
loss = loss_fn(logits, targets)

loss

tensor(10.9064, grad_fn=<NllLossBackward0>)

## Задание 1
Реализуйте отдельные этапы генерации на маленькой модели `Qwen/Qwen2.5-0.5B`. Для ускорения работы останавливайте генерацию по достижении max_new_tokens токенов. 

Токенизатор и модель уже загружены. Для примера попросите сгенерировать продолжение предложения «Привет,».


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")

def generate(model, tokenizer, prompt, max_new_tokens=30):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    generated = input_ids

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(generated)
            next_token_logits = outputs.logits[:, -1, :] 
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            generated = torch.cat((generated, next_token), dim=1)

    return tokenizer.decode(generated[0], skip_special_tokens=True)

print(generate(model, tokenizer, 'Привет,'))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 14212.17it/s]


Привет, я новичок в программировании, и вот у меня возникла проблема с вводом чисел в консоль. Я


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen1.5-0.5B')

messages = [
    {"role": "system", "content": "Ты дружелюбный помощник."},
    {"role": "user", "content": "Расскажи анекдот"},
    {"role": "assistant", "content": "Программист — это машина для преобразования кофе в код."}
]

prompt = tokenizer.apply_chat_template(
                                      messages, 
                                      tokenize=False
# не надо сразу токенизировать промпт, чтобы посмотреть на текстовую версию
) 
print(prompt)

<|im_start|>system
Ты дружелюбный помощник.<|im_end|>
<|im_start|>user
Расскажи анекдот<|im_end|>
<|im_start|>assistant
Программист — это машина для преобразования кофе в код.<|im_end|>



In [4]:
print(tokenizer.chat_template) 

{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful assistant<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


# Генерация текста с помощью библиотеки transformers

Мы разобрали все аспекты генерации и переходим к практике — генерируем текст с помощью библиотеки transformers.

Ручная генерация в продакшен-сервисах неэффективна и неудобна — автоматизируем её посредством API .generate(), который подходит даже для жадной генерации ([подробнее](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/text_generation#transformers.GenerationMixin.generate)).

## Примеры

Во всех примерах будем использовать модель `Qwen/Qwen2.5-0.5B-Instruct`, которая обучалась в режиме чата (об этом говорит суффикс instruct), но продолжать текст она тоже умеет. Давайте это проверим.

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

text = 'Привет'
inputs = tokenizer(
    text,
    return_tensors="pt",
)
print(inputs.input_ids.shape)
# torch.Size([1, 3])

outputs = model.generate(**inputs, 
                         max_new_tokens=10, # ограничим генерацию до 10 токенов
                         do_sample=False) # детерминированная генерация
# по умолчанию генерация остановится при достижении tokenizer.eos_token
print(tokenizer.batch_decode(outputs))
# ['Привет, у меня есть 2 таблицы: \n\n']

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 7236.76it/s]


torch.Size([1, 3])
['Привет, у меня есть 2 таблицы: \n\n']


Обратите внимание:
- функция .generate принимает на вход батч, причём не только размером 1.
- do_sample=False помогает воспроизвести этот код с таким же результатом.

А что же формат чата? Токенизатор преобразует чат в текст со специальными токенами, и в модель подаются те же числовые токены.

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

messages = [
    {"role": "user", "content": "Ты кто?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True, # уже токенизируем, чтобы подать в модель
    return_dict=True,
    return_tensors="pt",
)
print(inputs.input_ids.shape)
# torch.Size([1, 33])

outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.batch_decode(outputs))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 13474.56it/s]


torch.Size([1, 33])
['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nТы кто?<|im_end|>\n<|im_start|>assistant\nЯ искусственный интеллект созданный компанией Alibaba Cloud. Я могу помочь вам с любыми вопросами или задачами, связанными с искусственным интеллектом, машинным обучением, разработкой программных продуктов и многое другое.<|im_end|>']


Ответ модели следует за `<|im_start|>assistant\n`. Здесь специальные токены отличаются от тех, что вам уже знакомы, но вы можете предположить их предназначение, например, `<|im_end|>`, скорее всего, означает конец реплики.

В примере выше много однотипного кода и параметров для выполнения токенизации, вызова модели и декодирования ответа. На этот случай в transformers есть отдельный инструмент `pipeline`:

In [8]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
messages = [
    {"role": "user", "content": "Кто ты?"},
]
print(pipe(messages))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 14622.03it/s]
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'Кто ты?'}, {'role': 'assistant', 'content': 'Я - это искусственный интеллект, созданный компанией Alibaba Cloud. Я способен обмениваться информацией и предоставлять помощь в различных областях.'}]}]


На вход можно подавать только строку:

In [9]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
text = 'Привет'
pipe(text, max_new_tokens=10)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 19133.09it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Привет, как можно узнать информацию о том, где находится'}]

Напишите функцию выбора следующего токена: 
- Примените температуру, фильтры top‑k и top‑p и штраф повторов.
- Используйте функцию в простом цикле генерации поверх небольшой модели.

Как и в предыдущем уроке, ориентируйтесь на PyTorch и библиотеку transformers. В качестве модели используйте Qwen.

Внимательно прочитайте комментарии в прекоде — они описывают детали реализации, которые вам нужно дополнить. Выполните задание локально, в своём окружении, а затем сверьтесь с авторским решением. 

In [10]:
import torch
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Для воспроизводимости результатов можно зафиксировать сид
torch.manual_seed(0)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model.eval()


@torch.no_grad()
def sample_next_token(logits, generated_ids, temperature=1.0, top_k=None, top_p=None, repetition_penalty=1.0):
    # 1) Применяем штраф повторов: уменьшаем логиты для уже встречавшихся токенов
    if (repetition_penalty is not None and 
            repetition_penalty > 1.0 and
            generated_ids is not None and
            generated_ids.numel() > 0):
        # Считаем частоты появлений токенов в истории
        vals, counts = torch.unique(generated_ids, return_counts=True)
        logits = logits.clone()
        # Чем чаще встречался токен, тем сильнее уменьшаем логит
        for v, c in zip(vals, counts):
            # делим логит на коэффициент в степени количества повторов: 
            # 1) используйте параметр repetition_penalty, 
            # 2) возведите его в степень поличества повторов
            # 3) поделите исходный логит на получившееся число
            logits[..., v] /= (repetition_penalty ** c.item())

    # 2) Температура: масштабируем логиты
    temp = max(1e-6, float(temperature))
    logits /= temp # делим логиты на температуру

    # 3) Фильтр top-k: оставляем k самых вероятных вариантов
    if top_k is not None and top_k > 0:
        # выбираем топ k токенов и их логитов
        kth =  torch.topk(logits, k=top_k)[0][..., -1, None] # используйте torch.topk и параметр top_k
        mask = logits < kth
        logits = logits.masked_fill(mask, float('-inf'))

    # 4) Фильтр top-p (nucleus): динамически находим минимальный префикс по суммарной вероятности
    if top_p is not None and 0.0 < top_p < 1.0:
        probs = F.softmax(logits, dim=-1)
        # сортируем токены по убыванию вероятностей
        sorted_probs, sorted_idx = torch.sort(probs, descending=True) # используйте torch.sort
        # считаем для каждого индекса сумму всех предыдущих      
        cumprobs = torch.cumsum(sorted_probs, dim=-1) # используйте torch.cumsum
        # Оставляем только те, что входят в минимальный набор с суммой ≥ p
        keep_mask = cumprobs <= top_p
        # Обязательно оставим хотя бы самый вероятный токен
        keep_mask[..., 0] = True
        filtered = torch.full_like(sorted_probs, float('-inf'))
        filtered[keep_mask] = torch.log(sorted_probs[keep_mask]) # логарифмируем, потому что дальше снова считаем softmax: используйте torch.log
        # Возвращаемся к исходному порядку словаря
        logits = torch.full_like(logits, float('-inf'))
        logits.scatter_(-1, sorted_idx, filtered)  # размещаем новые значения

    # 5) Сэмпл из полученного распределения
    probs = F.softmax(logits, dim=-1)
    # Выведем топ-5 токенов и их вероятности перед выбором
    top_probs, top_ids = torch.topk(probs, k=5) # используйте torch.topk для топ-5 токенов с вероятностями
    print("Top-5 candidates:")
    for pid, pval in zip(top_ids.tolist(), top_probs.tolist()):
        print(f"  {tokenizer.decode([pid]):<15} : {pval:.4f}")
    # выбор токена с учётом вероятности
    next_id = torch.multinomial(probs, 1) # используйте torch.multinomial для семплирования

    return next_id


@torch.no_grad()
def generate_custom(prompt, max_new_tokens=40, temperature=0.9, top_k=20, top_p=0.9, repetition_penalty=1.1):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        outputs = model(input_ids=generated)
        next_logits = outputs.logits[:, -1, :].squeeze(0)
        next_id = sample_next_token(
            next_logits, generated_ids=generated[0],
            temperature=temperature, top_k=top_k, top_p=top_p, repetition_penalty=repetition_penalty
        )
        generated = torch.cat([generated, next_id.unsqueeze(0)], dim=1)
        # Остановимся по EOS, если он определён у токенизатора
        if tokenizer.eos_token_id is not None and next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(generated[0])


print(generate_custom("Придумай смешной, но вежливый тост про разработчиков:", max_new_tokens=10))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 6361.89it/s]


Top-5 candidates:
   "              : 1.0000
  #               : 0.0000
  %               : 0.0000
  !               : 0.0000
  "               : 0.0000
Top-5 candidates:
  В               : 0.1904
  К               : 0.1230
  И               : 0.1021
  М               : 0.0903
  Д               : 0.0703
Top-5 candidates:
  от              : 0.8320
  с               : 0.1060
  се              : 0.0640
  #               : 0.0000
  %               : 0.0000
Top-5 candidates:
   как            : 0.1396
  ,               : 0.1318
   такой          : 0.1250
   так            : 0.1162
   что            : 0.1025
Top-5 candidates:
   дел            : 0.2236
   я              : 0.1738
   такое          : 0.1533
   мы             : 0.1123
  ,               : 0.0728
Top-5 candidates:
   сделал         : 0.1650
   дел            : 0.1465
   хочу           : 0.0947
   могу           : 0.0947
   д              : 0.0613
Top-5 candidates:
  стро            : 0.2598
   них            : 0.1777
   самом  

## Сэмплинг
Стохастический сэмплинг, напротив, не хранит пачку гипотез. Его главное свойство — случайность, что следует из названия («стохастический»). Сэмплинг на каждом шаге берёт одно случайное решение из отфильтрованного и «темперированного» распределения и движется дальше. Управляя температурой top‑k и top‑p, мы делаем этот процесс более-менее смелым. Если нужно несколько альтернатив, можно сделать несколько прогонов и выбрать самый удачный по внешним метрикам или субъективной оценке. Сэмплинг дешевле и проще, он лучше работает там, где единственного правильного ответа не существует.

Чтобы почувствовать разницу, применим поочерёдно оба режима генерации к одной и той же модели и промпту. 

Начнём с beam search: зададим число лучей, включим раннюю остановку по достижению конца последовательности. 

Затем переключимся на сэмплинг: включим do_sample, настройки температуры и top‑p. Заметим, что при do_sample=False следующий токен выбирается жадной стратегией (через argmax).

Пример использования стратегии beam search:

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

messages = [{"role": "user", "content": "Сформулируй краткий и точный ответ: что такое байесовская вероятность?"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")

out_beam = model.generate(
    inputs["input_ids"], 
    max_new_tokens=100,
    num_beams=4, 
    early_stopping=True,
    length_penalty=0.8, 
    do_sample=False
)
print(tokenizer.batch_decode(out_beam))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 13436.01it/s]


['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nСформулируй краткий и точный ответ: что такое байесовская вероятность?<|im_end|>\n<|im_start|>assistant\nБайесовская вероятность (Bayesian probability) - это математическое моделирование, используемое для оценки вероятностей. Она основана на теории байесовской вероятности, которая предполагает, что вероятности могут изменяться в зависимости от информации, которую мы знаем или не знаем. Байесовская вероятность используется в различных областях, в']


In [16]:
messages = [{"role": "user", "content": "Придумай три необычных слогана кофейни, играя словами со словом ‘байес’."}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")

out_sample = model.generate(
    inputs["input_ids"], 
    max_new_tokens=100,
    do_sample=True, 
    temperature=0.9,
    top_p=0.9, 
    top_k=50, 
    repetition_penalty=1.1
)
print(tokenizer.batch_decode(out_sample))

['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nПридумай три необычных слогана кофейни, играя словами со словом ‘байес’.<|im_end|>\n<|im_start|>assistant\nКонечно! Вот три необычных слогана, придуманные на основе слова "байес":\n\n1. **Байес-Салат** - это легендарное кафе в Нью-Йорке, которое известно своими элегантными стилами и богатыми блюдами.\n\n2. **Байес-Айленд** - это одноимённый булгалистический ароматический нап']
